# LightGBM tuning

In [1]:
%pip install -q "mlflow==2.22.0" "sqlalchemy==2.0.41"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
grpcio-tools 1.85.1 requires protobuf<8.0.0,>=7.35.1, but you have protobuf 6.33.6 which is incompatible.
nilearn 0.14.1 requires packaging>=26.0.0, but you have packaging 24.2 which is incompatible.
datasets 4.8.5 requires pyarrow>=21.0.0, but you have pyarrow 19.0.1 which is incompatible.
bigframes 2.51.0 requires pyarrow>=23.0.1, but you have pyarrow 19.0.1 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path

import lightgbm as lgb
import mlflow
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

output = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
mlflow.set_tracking_uri(f"sqlite:///{output / 'lightgbm_mlflow.db'}")
mlflow.set_experiment("fertilizer-lightgbm")

if Path("/kaggle/input").exists():
    candidates = [p for p in Path("/kaggle/input").rglob("train.csv")
                  if "Fertilizer Name" in pd.read_csv(p, nrows=0).columns]
    train_path, = candidates
else:
    train_path = next(p for p in (Path("data/train.csv"), Path("../data/train.csv")) if p.exists())

train = pd.read_csv(train_path).dropna()
numeric = ["Temparature", "Humidity", "Moisture", "Nitrogen", "Potassium", "Phosphorous"]
categorical = ["Soil Type", "Crop Type"]
X = train[numeric + categorical]
y = LabelEncoder().fit_transform(train["Fertilizer Name"])
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
encoder = ColumnTransformer([
    ("numeric", "passthrough", numeric),
    ("category", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32), categorical),
])
X_train = encoder.fit_transform(X_train)
X_val = encoder.transform(X_val)
print(f"{len(X_train):,} training rows, {len(X_val):,} validation rows")

2026/11/23 17:11:01 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/23 17:11:01 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running upgrade  -> 451aebb31d03, add metric step
INFO  [alembic.runtime.migration] Running upgrade 451aebb31d03 -> 90e64c465722, migrate user column to tags
INFO  [alembic.runtime.migration] Running upgrade 90e64c465722 -> 181f10493468, allow nulls for metric values
INFO  [alembic.runtime.migration] Running upgrade 181f10493468 -> df50e92ffc5e, Add Experiment Tags Table
INFO  [alembic.runtime.migration] Running upgrade df50e92ffc5e -> 7ac759974ad8, Update run tags with larger limit
INFO  [alembic.runtime.migration] Running upgrade 7ac759974ad8 -> 89d4b8295536, create latest metrics table
INFO  [89d4b8295536_create_latest_metrics_table_py] Migration complete!
INFO  

600,000 training rows, 150,000 validation rows


## Depth and leaves
Fix the model seed at 42. Stop on validation log loss.

In [3]:
def evaluate(depth, leaves, seed):
    model = lgb.LGBMClassifier(
        device_type="gpu",max_bin=63,
        max_depth=depth, num_leaves=leaves, random_state=seed,
        n_estimators=1000, learning_rate=0.08, reg_lambda=1.0,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbosity=-1,
    )
    with mlflow.start_run(run_name=f"lgbm_depth{depth}_leaves{leaves}_seed{seed}") as run:
        mlflow.log_params(model.get_params())
        mlflow.log_params({"split_seed": 42, "early_stopping_rounds": 50,
                           "train_rows": len(y_train), "validation_rows": len(y_val)})
        model.fit(
            X_train, y_train, eval_set=[(X_val, y_val)], eval_metric="multi_logloss",
            callbacks=[lgb.early_stopping(50, verbose=False)],
        )
        proba = model.predict_proba(X_val)
        top3 = np.argsort(-proba, axis=1)[:, :3]
        map3 = ((top3 == y_val[:, None]) / np.arange(1, 4)).sum(axis=1).mean()
        loss = log_loss(y_val, proba)
        mlflow.log_metrics({"val_map3": map3, "val_logloss": loss, "best_round": model.best_iteration_})
        return dict(depth=depth, leaves=leaves, seed=seed, rounds=model.best_iteration_,
                    map3=map3, logloss=loss, run_id=run.info.run_id)


grid = [(4, 8), (4, 16), (6, 31), (6, 63), (8, 63), (8, 127)]
runs = [evaluate(depth, leaves, 42) for depth, leaves in grid]
sweep = pd.DataFrame(runs).sort_values(["map3", "depth", "leaves"], ascending=[False, True, True])
sweep

1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature 

,depth,leaves,seed,rounds,map3,logloss,run_id
3,6,63,42,504,0.337902,1.914021,61ebd68083c34bc7b8b902fd1654ff49
2,6,31,42,870,0.337506,1.914523,a3ac65871d824873b1a937ab3d12cc42
5,8,127,42,242,0.335874,1.915893,626dd03e4afd44c4a383bd1bceafb3c2
1,4,16,42,1000,0.335854,1.915104,b426be6ec8bf4e1fb50e9d7f9b192673
4,8,63,42,452,0.334421,1.916075,473ebb2793214961b735b5860bcc7e73
0,4,8,42,1000,0.330242,1.919154,3e49039118274f25960f3fdb71c01202


## Seed comparison
Keep the split fixed. XGBoost scores come from the previous depth-6 run.

In [4]:
depth, leaves = sweep.iloc[0][["depth", "leaves"]].astype(int)
seed_runs = [evaluate(depth, leaves, seed) for seed in [2025, 7]]
results = pd.DataFrame(runs + seed_runs)
results.to_csv(output / "lightgbm_results.csv", index=False)

scores = results.loc[(results.depth == depth) & (results.leaves == leaves)].set_index("seed")
comparison = scores[["map3"]].rename(columns={"map3": "lightgbm"})
comparison["xgboost"] = pd.Series({42: 0.340220, 2025: 0.3411911111111111, 7: 0.34010888888888885})
comparison["delta"] = comparison.lightgbm - comparison.xgboost
comparison.loc["mean"] = comparison.mean()
comparison

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,lightgbm,xgboost,delta
seed,,,
42,0.337902,0.340220,-0.002318
2025,0.337804,0.341191,-0.003387
7,0.337208,0.340109,-0.002901
mean,0.337638,0.340507,-0.002869
